# Plot C: within-category kNN diversity

**Purpose:** Recreate the kNN diversity analysis for CCN 2026 submission figures.

For each category, this notebook computes mean distance to k nearest neighbors within category and summarizes micro-structure (lower = tighter local clusters, higher = more uniform spread).

**Outputs:**
1. Per-category kNN diversity summary CSV.
2. Optional per-exemplar kNN distance CSV.

## Parameters

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
from tqdm import tqdm

SCRIPT_DIR = Path(".").resolve()

embeddings_to_run = ["clip", "dinov3"]
k_list = [5]         # e.g. [5, 10] for multi-k summary
save_exemplar_csv = False
out_dir = SCRIPT_DIR / "plotC_knn_diversity_outputs"

# Plot B output directory override.
# - None: auto-select latest plotB_tsne_distance_to_centroid_outputs_* directory
# - Path(...): force a specific Plot B output directory
plotb_dir_override = None

## Load BV embeddings (reuse from centroid script)

We import shared BV embedding-loading utilities used in the centroid analyses; if unavailable (e.g. running this notebook alone), we define a minimal local loader and config.

In [ ]:
try:
    from bv_to_things_centroid_distances import (
        load_bv_embeddings,
        EXCLUDED_SUBJECT,
        MIN_EXEMPLARS,
    )
except ImportError:
    FILTER_THRESHOLD = 0.27
    GROUPED_EMBEDDINGS_BASE = Path(os.getenv("BV_EMBEDDINGS_BASE", "SET_BV_EMBEDDINGS_BASE")).expanduser()
    GROUPED_EMBEDDINGS_DIRS = {
        "clip": GROUPED_EMBEDDINGS_BASE / f"clip_embeddings_grouped_by_age-mo_filtered-{FILTER_THRESHOLD}_normalized",
        "dinov3": GROUPED_EMBEDDINGS_BASE / f"dinov3_embeddings_grouped_by_age-mo_filtered-{FILTER_THRESHOLD}_normalized",
    }
    EXCLUDED_SUBJECT = "00270001"
    MIN_EXEMPLARS = 2

    def load_valid_exemplar_keys(per_file_precision_csv, threshold=0.6):
        if per_file_precision_csv is None or not Path(per_file_precision_csv).exists():
            return None
        df = pd.read_csv(per_file_precision_csv, usecols=["filename", "class", "precision"])\
            .dropna(subset=["filename", "class", "precision"]).copy()
        df["class"] = df["class"].astype(str).str.strip().str.lower()
        df["precision"] = pd.to_numeric(df["precision"], errors="coerce")
        df = df[df["precision"] > threshold].copy()
        valid = {}
        for fname_raw, cat_raw, _prec in df[["filename", "class", "precision"]].itertuples(index=False, name=None):
            fname = Path(str(fname_raw).strip()).name
            cat = str(cat_raw).strip().lower()
            stem = Path(fname).stem
            remainder = stem
            prefix = f"{cat}_"
            if stem.startswith(prefix):
                parts = stem.split("_", 2)
                if len(parts) == 3:
                    remainder = parts[2]
            entry = valid.setdefault(cat, {"full": set(), "remainder": set()})
            entry["full"].add(stem)
            entry["remainder"].add(remainder)
        return valid


    def is_valid_exemplar(cat_name, stem, valid_exemplar_keys):
        if valid_exemplar_keys is None:
            return True
        cat = str(cat_name).strip().lower()
        entry = valid_exemplar_keys.get(cat)
        if not entry:
            return False
        s = str(stem).strip()
        if s in entry["full"] or s in entry["remainder"]:
            return True
        base = Path(s).stem
        return (base in entry["full"]) or (base in entry["remainder"])


    def load_bv_embeddings(embedding_type, allowed_categories, excluded_subject, min_exemplars=2, valid_exemplar_keys=None):
        grouped_dir = GROUPED_EMBEDDINGS_DIRS[embedding_type]
        if not grouped_dir.exists():
            raise FileNotFoundError(f"BV grouped dir not found: {grouped_dir}")
        category_embeddings = {}
        category_exemplar_ids = {}
        for cat_folder in sorted(grouped_dir.iterdir()):
            if not cat_folder.is_dir():
                continue
            cat_name = cat_folder.name
            if allowed_categories is not None and cat_name not in allowed_categories:
                continue
            embs, ids = [], []
            for f in cat_folder.glob("*.npy"):
                stem = f.stem
                parts = stem.split("_")
                if len(parts) < 2:
                    continue
                subject_id, age_mo = parts[0], None
                try:
                    age_mo = int(parts[1])
                except ValueError:
                    continue
                if excluded_subject and subject_id == excluded_subject:
                    continue
                if not is_valid_exemplar(cat_name, stem, valid_exemplar_keys):
                    continue
                try:
                    e = np.load(f)
                    e = np.asarray(e, dtype=np.float64).flatten()
                    embs.append(e)
                    ids.append((subject_id, age_mo))
                except Exception:
                    continue
            if len(embs) >= min_exemplars:
                category_embeddings[cat_name] = np.array(embs)
                category_exemplar_ids[cat_name] = ids
        return category_embeddings, category_exemplar_ids

# Match preprint-2026 category selection (129 categories)
CATEGORIES_FILE = SCRIPT_DIR / "../../data/included_categories.txt"
PER_CLASS_VALIDATION_CSV = SCRIPT_DIR / "../../annotation/per_class_validation_data.csv"
PER_FILE_PRECISION_CSV = SCRIPT_DIR / "../../annotation/per_file_precision_data.csv"
PRECISION_THRESHOLD = 0.6

def load_valid_exemplar_keys(per_file_precision_csv, threshold=0.6):
    if per_file_precision_csv is None or not Path(per_file_precision_csv).exists():
        return None
    df = pd.read_csv(per_file_precision_csv, usecols=["filename", "class", "precision"])\
        .dropna(subset=["filename", "class", "precision"]).copy()
    df["class"] = df["class"].astype(str).str.strip().str.lower()
    df["precision"] = pd.to_numeric(df["precision"], errors="coerce")
    df = df[df["precision"] > threshold].copy()
    valid = {}
    for fname_raw, cat_raw, _prec in df[["filename", "class", "precision"]].itertuples(index=False, name=None):
        fname = Path(str(fname_raw).strip()).name
        cat = str(cat_raw).strip().lower()
        stem = Path(fname).stem
        remainder = stem
        prefix = f"{cat}_"
        if stem.startswith(prefix):
            parts = stem.split("_", 2)
            if len(parts) == 3:
                remainder = parts[2]
        entry = valid.setdefault(cat, {"full": set(), "remainder": set()})
        entry["full"].add(stem)
        entry["remainder"].add(remainder)
    return valid


def is_valid_exemplar(cat_name, stem, valid_exemplar_keys):
    if valid_exemplar_keys is None:
        return True
    cat = str(cat_name).strip().lower()
    entry = valid_exemplar_keys.get(cat)
    if not entry:
        return False
    s = str(stem).strip()
    if s in entry["full"] or s in entry["remainder"]:
        return True
    base = Path(s).stem
    return (base in entry["full"]) or (base in entry["remainder"])


def load_allowed_categories():
    allowed = None
    if CATEGORIES_FILE.exists():
        with open(CATEGORIES_FILE) as f:
            allowed = set(line.strip().lower() for line in f if line.strip())
    if not PER_CLASS_VALIDATION_CSV.exists():
        return allowed
    val_df = pd.read_csv(PER_CLASS_VALIDATION_CSV, usecols=["class", "precision"])\
        .dropna(subset=["class", "precision"]).copy()
    val_df["class"] = val_df["class"].astype(str).str.strip().str.lower()
    val_df["precision"] = pd.to_numeric(val_df["precision"], errors="coerce")
    precision_allowed = set(val_df.loc[val_df["precision"] > PRECISION_THRESHOLD, "class"])
    if allowed is None:
        return precision_allowed
    return allowed & precision_allowed


# Month-level grouped embedding stems (e.g., subject_age_month_level_avg) do not align
# with crop-level per-file precision stems, so keep this disabled for Plot C.
USE_PER_FILE_STEM_FILTER = False
if USE_PER_FILE_STEM_FILTER:
    valid_exemplar_keys = load_valid_exemplar_keys(PER_FILE_PRECISION_CSV, threshold=PRECISION_THRESHOLD)
else:
    valid_exemplar_keys = None

## kNN mean distance per exemplar

For each row in X we compute mean L2 distance to its k nearest neighbors (excluding self), using k+1 neighbors and dropping the first (self).

In [ ]:
def compute_knn_mean_distances(X, k):
    from sklearn.neighbors import NearestNeighbors
    n = X.shape[0]
    n_neighbors = min(k + 1, n)
    if n_neighbors < 2:
        return np.full(n, np.nan)
    nn = NearestNeighbors(n_neighbors=n_neighbors, metric="euclidean", algorithm="auto")
    nn.fit(X)
    distances, _ = nn.kneighbors(X)
    mean_knn = np.mean(distances[:, 1:], axis=1)
    return mean_knn


def run_knn_per_category(bv_embeddings, bv_ids, categories, k):
    summary_rows = []
    exemplar_rows = []
    for cat in tqdm(categories, desc="kNN per category"):
        X = bv_embeddings[cat]
        id_list = bv_ids[cat]
        n = X.shape[0]
        effective_k = min(k, n - 1)
        if effective_k < 1:
            summary_rows.append({
                "category": cat, "n_exemplars": n, "k": k, "effective_k": 0,
                "mean_knn_dist": np.nan, "std_knn_dist": np.nan, "median_knn_dist": np.nan,
            })
            continue
        mean_knn = compute_knn_mean_distances(X, effective_k)
        summary_rows.append({
            "category": cat, "n_exemplars": n, "k": k, "effective_k": effective_k,
            "mean_knn_dist": float(np.nanmean(mean_knn)),
            "std_knn_dist": float(np.nanstd(mean_knn)),
            "median_knn_dist": float(np.nanmedian(mean_knn)),
        })
        for i, (sid, age_mo) in enumerate(id_list):
            exemplar_rows.append({
                "category": cat, "subject_id": sid, "age_mo": age_mo,
                "mean_knn_dist": float(mean_knn[i]),
            })
    return summary_rows, exemplar_rows

## Run for each k and save

In [ ]:
out_dir = Path(out_dir)
out_dir.mkdir(parents=True, exist_ok=True)

allowed = load_allowed_categories()
print(f"Using {len(allowed) if allowed else 'all'} categories")

summary_columns = ["category", "n_exemplars", "k", "effective_k", "mean_knn_dist", "std_knn_dist", "median_knn_dist"]
summary_by_embedding = {}
for embedding in embeddings_to_run:
    print(f"\n=== Running embedding: {embedding} ===")
    print("Loading BV embeddings...")
    import inspect
    sig = inspect.signature(load_bv_embeddings)
    kwargs = dict(allowed_categories=allowed, excluded_subject=EXCLUDED_SUBJECT, min_exemplars=MIN_EXEMPLARS)
    if "valid_exemplar_keys" in sig.parameters:
        kwargs["valid_exemplar_keys"] = valid_exemplar_keys
    bv_embeddings, bv_ids = load_bv_embeddings(embedding, **kwargs)
    categories = sorted(bv_embeddings.keys())
    print(f"Categories: {len(categories)}")

    all_summary = []
    for k in k_list:
        prefix = f"bv_within_category_knn_{embedding}_k{k}"
        summary_rows, exemplar_rows = run_knn_per_category(bv_embeddings, bv_ids, categories, k)
        summary_df = pd.DataFrame(summary_rows)
        if summary_df.empty:
            summary_df = pd.DataFrame(columns=summary_columns)
        else:
            summary_df = summary_df.sort_values("mean_knn_dist", ascending=True).reset_index(drop=True)

        summary_path = out_dir / f"{prefix}_summary.csv"
        summary_df.to_csv(summary_path, index=False)
        print(f"Saved {prefix}: {summary_path}")

        if save_exemplar_csv and exemplar_rows:
            exemplar_df = pd.DataFrame(exemplar_rows)
            exemplar_df["rank_within_cat"] = exemplar_df.groupby("category")["mean_knn_dist"].rank(
                method="first", ascending=True
            ).astype(int)
            exemplar_path = out_dir / f"{prefix}_per_exemplar.csv"
            exemplar_df.to_csv(exemplar_path, index=False)
            print(f"Saved per-exemplar: {exemplar_path}")

        all_summary.append(summary_df.assign(k_used=k, embedding=embedding))

    if all_summary:
        combined = pd.concat(all_summary, ignore_index=True)
    else:
        combined = pd.DataFrame(columns=summary_columns + ["k_used", "embedding"])

    if len(k_list) > 1:
        combined_path = out_dir / f"bv_within_category_knn_{embedding}_multi_k_summary.csv"
        combined.to_csv(combined_path, index=False)
        print(f"Saved combined multi-k summary: {combined_path}")

    last_summary = all_summary[-1].drop(columns=["k_used", "embedding"], errors="ignore") if all_summary else pd.DataFrame(columns=summary_columns)
    summary_by_embedding[embedding] = {
        "combined": combined,
        "last_summary": last_summary,
    }

embedding = embeddings_to_run[0]
print("Done. Lower mean_knn_dist = more micro-structure; higher = more uniform spread.")

## Visualize category diversity rank (high to low)

Create rank-based plots so categories are explicitly ordered from highest diversity (`mean_knn_dist`) to lowest diversity.

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

# Use the summary from the last processed k by default.
if len(k_list) > 1:
    plot_df = all_summary[-1].copy()
else:
    plot_df = summary_df.copy()

plot_df = plot_df.dropna(subset=["mean_knn_dist"]).copy()
if plot_df.empty:
    raise RuntimeError(
        "No categories available for plotting. Check allowed-category and exemplar filters "
        "(e.g., precision threshold, excluded subject, MIN_EXEMPLARS)."
    )

plot_df = plot_df.sort_values("mean_knn_dist", ascending=False).reset_index(drop=True)
plot_df["rank_high_to_low"] = np.arange(1, len(plot_df) + 1)

k_used = int(plot_df["k"].iloc[0]) if "k" in plot_df.columns else int(k_list[-1])

fig, axes = plt.subplots(1, 2, figsize=(18, max(10, len(plot_df) * 0.18)), constrained_layout=True)

# Left: full ranked horizontal bar chart (high -> low diversity)
axes[0].barh(plot_df["category"], plot_df["mean_knn_dist"], color="steelblue")
axes[0].invert_yaxis()
axes[0].set_xlabel("Mean kNN distance")
axes[0].set_ylabel("Category")
axes[0].set_title(f"Category diversity rank (high to low), {embedding.upper()} k={k_used}")

# Right: rank curve
axes[1].plot(plot_df["rank_high_to_low"], plot_df["mean_knn_dist"], marker="o", linewidth=1.5, markersize=3)
axes[1].set_xlabel("Rank (1 = highest diversity)")
axes[1].set_ylabel("Mean kNN distance")
axes[1].set_title("Diversity by rank")
axes[1].grid(alpha=0.3)

plt.show()

# Optional: save figure for manuscript/slide reuse
fig_path = out_dir / f"bv_within_category_knn_{embedding}_k{k_used}_rank_high_to_low.png"
pdf_path = fig_path.with_suffix(".pdf")
fig.savefig(fig_path, dpi=300, bbox_inches="tight")
fig.savefig(pdf_path, bbox_inches="tight")
print(f"Saved rank figure: {fig_path}")
print(f"Saved rank PDF: {pdf_path}")

# Show top/bottom ranked categories
top_n = 15
display_cols = ["rank_high_to_low", "category", "mean_knn_dist", "n_exemplars", "effective_k"]
print(f"\nTop {top_n} highest-diversity categories:")
display(plot_df[display_cols].head(top_n))

print(f"\nTop {top_n} lowest-diversity categories:")
display(plot_df[display_cols].tail(top_n).sort_values("rank_high_to_low", ascending=False))

## Combined extremes plots (Top/Bottom 5 and 10)

Side-by-side plots to compare the most and least diverse categories at a glance.

In [ ]:
# Assumes `plot_df` already exists from the previous visualization cell.
# If not, rebuild from the latest summary.
if "plot_df" not in globals():
    if len(k_list) > 1:
        plot_df = all_summary[-1].copy()
    else:
        plot_df = summary_df.copy()
    plot_df = plot_df.dropna(subset=["mean_knn_dist"]).copy()
    plot_df = plot_df.sort_values("mean_knn_dist", ascending=False).reset_index(drop=True)
    plot_df["rank_high_to_low"] = np.arange(1, len(plot_df) + 1)


def make_extremes_df(df, n):
    top = df.head(n).copy()
    top["group"] = f"Top {n}"

    bottom = df.tail(n).copy().sort_values("mean_knn_dist", ascending=True)
    bottom["group"] = f"Bottom {n}"

    extremes = pd.concat([top, bottom], ignore_index=True)
    return extremes


def plot_extremes(ax, df, n):
    extremes = make_extremes_df(df, n)
    labels = [f"{r.category} (#{int(r.rank_high_to_low)})" for r in extremes.itertuples()]
    colors = ["#1f77b4" if g.startswith("Top") else "#d62728" for g in extremes["group"]]

    ax.barh(labels, extremes["mean_knn_dist"], color=colors)
    ax.invert_yaxis()
    ax.set_title(f"Top {n} and Bottom {n} Diversity")
    ax.set_xlabel("Mean kNN distance")

    # Add a divider between Top and Bottom blocks
    ax.axhline(y=n - 0.5, color="black", linestyle="--", linewidth=1, alpha=0.6)


fig, axes = plt.subplots(1, 2, figsize=(18, 10), constrained_layout=True)
plot_extremes(axes[0], plot_df, n=5)
plot_extremes(axes[1], plot_df, n=10)

k_used = int(plot_df["k"].iloc[0]) if "k" in plot_df.columns else int(k_list[-1])
fig.suptitle(f"Category diversity extremes, {embedding.upper()} k={k_used}", fontsize=14)

plt.show()

extremes_path = out_dir / f"bv_within_category_knn_{embedding}_k{k_used}_top_bottom_extremes_5_10.png"
extremes_pdf_path = extremes_path.with_suffix(".pdf")
fig.savefig(extremes_path, dpi=300, bbox_inches="tight")
fig.savefig(extremes_pdf_path, bbox_inches="tight")
print(f"Saved extremes figure: {extremes_path}")
print(f"Saved extremes PDF: {extremes_pdf_path}")

# Tabular view
ext_5 = make_extremes_df(plot_df, 5)[["group", "rank_high_to_low", "category", "mean_knn_dist", "n_exemplars", "effective_k"]]
ext_10 = make_extremes_df(plot_df, 10)[["group", "rank_high_to_low", "category", "mean_knn_dist", "n_exemplars", "effective_k"]]

print("\nTop/Bottom 5")
display(ext_5)
print("\nTop/Bottom 10")
display(ext_10)

## CLIP vs DINOv3 comparison

In [ ]:
if set(["clip", "dinov3"]).issubset(summary_by_embedding.keys()):
    # use k=5 if available, else first k in list
    k_for_compare = 5 if 5 in k_list else k_list[0]

    clip_df = summary_by_embedding["clip"]["combined"]
    clip_df = clip_df[clip_df["k_used"] == k_for_compare][["category", "mean_knn_dist", "n_exemplars"]].rename(
        columns={"mean_knn_dist": "clip_mean_knn_dist", "n_exemplars": "clip_n_exemplars"}
    )

    dino_df = summary_by_embedding["dinov3"]["combined"]
    dino_df = dino_df[dino_df["k_used"] == k_for_compare][["category", "mean_knn_dist", "n_exemplars"]].rename(
        columns={"mean_knn_dist": "dinov3_mean_knn_dist", "n_exemplars": "dinov3_n_exemplars"}
    )

    comparison_df = clip_df.merge(dino_df, on="category", how="inner")
    comparison_df["delta_knn_dinov3_minus_clip"] = (
        comparison_df["dinov3_mean_knn_dist"] - comparison_df["clip_mean_knn_dist"]
    )
    rank_clip = comparison_df["clip_mean_knn_dist"].rank(ascending=False, method="average")
    rank_dino = comparison_df["dinov3_mean_knn_dist"].rank(ascending=False, method="average")
    comparison_df["rank_shift_dinov3_minus_clip"] = rank_dino - rank_clip

    corr = comparison_df[["clip_mean_knn_dist", "dinov3_mean_knn_dist"]].corr().iloc[0, 1]

    cmp_path = out_dir / f"bv_within_category_knn_clip_vs_dinov3_k{k_for_compare}_comparison.csv"
    comparison_df.sort_values("delta_knn_dinov3_minus_clip", ascending=False).to_csv(cmp_path, index=False)
    print(f"Saved model comparison table: {cmp_path}")
    print(f"Category count: {len(comparison_df)}")
    print(f"Correlation (mean_knn_dist): {corr:.3f}")

    import matplotlib as mpl
    import matplotlib.pyplot as plt
    mpl.rcParams["pdf.fonttype"] = 42
    mpl.rcParams["ps.fonttype"] = 42
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), dpi=150)

    axes[0].scatter(comparison_df["clip_mean_knn_dist"], comparison_df["dinov3_mean_knn_dist"], s=18, alpha=0.8)
    mn = min(comparison_df["clip_mean_knn_dist"].min(), comparison_df["dinov3_mean_knn_dist"].min())
    mx = max(comparison_df["clip_mean_knn_dist"].max(), comparison_df["dinov3_mean_knn_dist"].max())
    axes[0].plot([mn, mx], [mn, mx], linestyle="--", linewidth=1)
    axes[0].set_xlabel("CLIP mean kNN distance")
    axes[0].set_ylabel("DINOv3 mean kNN distance")
    axes[0].set_title(f"Diversity agreement at k={k_for_compare} (r={corr:.2f})")

    axes[1].hist(comparison_df["delta_knn_dinov3_minus_clip"], bins=25, alpha=0.9)
    axes[1].axvline(0.0, linestyle="--", linewidth=1)
    axes[1].set_xlabel("DINOv3 - CLIP (mean kNN distance)")
    axes[1].set_ylabel("Category count")
    axes[1].set_title("Cross-model diversity shift")

    fig.tight_layout()
    fig_path = out_dir / f"bv_within_category_knn_clip_vs_dinov3_k{k_for_compare}_comparison.png"
    fig_pdf_path = fig_path.with_suffix(".pdf")
    fig.savefig(fig_path, bbox_inches="tight")
    fig.savefig(fig_pdf_path, bbox_inches="tight")
    plt.show()
    print(f"Saved comparison figure: {fig_path}")
    print(f"Saved comparison PDF: {fig_pdf_path}")
else:
    print("Skipping model comparison: need both clip and dinov3 in summary_by_embedding")

## Recreate CCN 2026 variability 2x2 panel

This recreates `ccn2026_variability_2x2_panel.png` by combining:
- global dispersion (Plot B: mean distance to BV centroid)
- local coherence (Plot C: mean kNN distance)
- CLIP and DINOv3 side-by-side
- rank-shift agreement/disagreement diagnostics

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

base_dir = Path(".").resolve()
out_dir = Path(out_dir)
out_dir.mkdir(parents=True, exist_ok=True)

if plotb_dir_override is None:
    candidates = sorted(base_dir.glob("plotB_tsne_distance_to_centroid_outputs_*"))
    if not candidates:
        raise FileNotFoundError(
            "No Plot B output directory found. Run notebook 02 first or set plotb_dir_override."
        )
    plotb_dir = candidates[-1]
else:
    plotb_dir = Path(plotb_dir_override)
    if not plotb_dir.is_absolute():
        plotb_dir = (base_dir / plotb_dir).resolve()

plotc_dir = out_dir

clip_knn_path = plotc_dir / "bv_within_category_knn_clip_k5_summary.csv"
dino_knn_path = plotc_dir / "bv_within_category_knn_dinov3_k5_summary.csv"

# Global variability for Plot C should always come from BV->BV centroid distance.
# Prefer explicit BV-centroid filenames; fall back to legacy names for compatibility.
clip_global_candidates = [
    # Current Plot B naming (2026-04): hyphenated prefix with _distance.
    plotb_dir / "bv-to-bv-centroid_distance_clip_summary.csv",
    # Backward-compatible fallback names.
    plotb_dir / "bv_to_bv_centroid_clip_summary.csv",
    plotb_dir / "bv_to_things_centroid_clip_summary.csv",
]
dino_global_candidates = [
    plotb_dir / "bv-to-bv-centroid_distance_dinov3_summary.csv",
    plotb_dir / "bv_to_bv_centroid_dinov3_summary.csv",
    plotb_dir / "bv_to_things_centroid_dinov3_summary.csv",
]

clip_global_path = next((p for p in clip_global_candidates if p.exists()), clip_global_candidates[0])
dino_global_path = next((p for p in dino_global_candidates if p.exists()), dino_global_candidates[0])

for p in [clip_knn_path, dino_knn_path, clip_global_path, dino_global_path]:
    if not p.exists():
        raise FileNotFoundError(f"Required input not found: {p}")

clip_knn = pd.read_csv(clip_knn_path)
dino_knn = pd.read_csv(dino_knn_path)
clip_global = pd.read_csv(clip_global_path)
dino_global = pd.read_csv(dino_global_path)

for name, df in [("clip", clip_global), ("dinov3", dino_global)]:
    if "mean_bv_to_bv_centroid" not in df.columns:
        raise ValueError(
            f"{name} global summary at {plotb_dir} is missing 'mean_bv_to_bv_centroid'. "
            "Notebook 03 expects BV->BV centroid metrics for global variability."
        )

print(f"Using Plot B outputs from: {plotb_dir}")
print(f"Global summaries: {clip_global_path.name}, {dino_global_path.name}")

clip_df = clip_knn.merge(
    clip_global[["category", "mean_bv_to_bv_centroid", "n_bv"]],
    on="category",
    how="inner",
)
clip_df["local_rank"] = clip_df["mean_knn_dist"].rank(ascending=False, method="average")
clip_df["global_rank"] = clip_df["mean_bv_to_bv_centroid"].rank(ascending=False, method="average")
clip_df["rank_shift_local_minus_global"] = clip_df["local_rank"] - clip_df["global_rank"]
clip_df["embedding"] = "clip"

# Keep n_bv only to avoid accidental mismatch from separate sources.
dino_df = dino_knn.merge(
    dino_global[["category", "mean_bv_to_bv_centroid", "n_bv"]],
    on="category",
    how="inner",
)
dino_df["local_rank"] = dino_df["mean_knn_dist"].rank(ascending=False, method="average")
dino_df["global_rank"] = dino_df["mean_bv_to_bv_centroid"].rank(ascending=False, method="average")
dino_df["rank_shift_local_minus_global"] = dino_df["local_rank"] - dino_df["global_rank"]
dino_df["embedding"] = "dinov3"

combined = pd.concat([clip_df, dino_df], ignore_index=True)
combined_path = out_dir / "ccn2026_local_global_extreme_categories_clip_dino.csv"
combined.to_csv(combined_path, index=False)

# 2x2 panel: cross-model and within-model comparisons (global dispersion first, then local coherence).
corr_df = clip_df[["category", "mean_knn_dist", "mean_bv_to_bv_centroid", "n_exemplars"]].rename(
    columns={
        "mean_knn_dist": "knn_clip",
        "mean_bv_to_bv_centroid": "global_clip",
        "n_exemplars": "n_exemplars_clip",
    }
).merge(
    dino_df[["category", "mean_knn_dist", "mean_bv_to_bv_centroid", "n_exemplars"]].rename(
        columns={
            "mean_knn_dist": "knn_dino",
            "mean_bv_to_bv_centroid": "global_dino",
            "n_exemplars": "n_exemplars_dino",
        }
    ),
    on="category",
    how="inner",
)
corr_df["n_exemplars"] = corr_df[["n_exemplars_clip", "n_exemplars_dino"]].mean(axis=1)

# Match notebooks 01/02: keep only per-file precision-valid exemplars.
repo_root = base_dir.parent.parent
per_file_precision_csv = repo_root / "annotation" / "per_file_precision_data.csv"
if not per_file_precision_csv.exists():
    raise FileNotFoundError(f"Per-file precision CSV not found: {per_file_precision_csv}")
precision_threshold_local = PRECISION_THRESHOLD if "PRECISION_THRESHOLD" in globals() else 0.6
pf_df = pd.read_csv(per_file_precision_csv, usecols=["filename", "class", "precision"]).dropna().copy()
pf_df["class"] = pf_df["class"].astype(str).str.strip().str.lower()
pf_df["precision"] = pd.to_numeric(pf_df["precision"], errors="coerce")
pf_df = pf_df[pf_df["precision"] > float(precision_threshold_local)].copy()
valid_counts = pf_df.groupby("class")["filename"].count().to_dict()
corr_df["n_valid_exemplars"] = corr_df["category"].astype(str).str.strip().str.lower().map(valid_counts).fillna(0).astype(int)
corr_df = corr_df[corr_df["n_valid_exemplars"] >= 2].copy()
if corr_df.empty:
    raise ValueError("No categories remain after per-file precision filtering for the correlation panel.")

# CDI semantic mapping + colors (shared with notebook 02).
repo_root = base_dir.parent.parent
semantic_csv = repo_root / "data" / "long_tailed_dist_prop_included_categories.csv"
if not semantic_csv.exists():
    raise FileNotFoundError(f"CDI semantic mapping not found: {semantic_csv}")

sem_df = pd.read_csv(semantic_csv, usecols=["category", "cdi_semantic"]).dropna(subset=["category", "cdi_semantic"])
sem_df["category"] = sem_df["category"].astype(str).str.strip()
sem_df["cdi_semantic"] = sem_df["cdi_semantic"].astype(str).str.strip().str.lower()

CDI_SEMANTIC_ORDER = [
    "animals", "body_parts", "clothing", "food_drink", "furniture_rooms",
    "household", "outside", "people", "toys", "vehicles", "other",
]
CDI_SEMANTIC_COLORS = {
    "animals": "#4DB8A8",
    "body_parts": "#E87A5F",
    "clothing": "#9B7EC8",
    "food_drink": "#E8A54C",
    "furniture_rooms": "#6BAB7A",
    "household": "#D97B9E",
    "outside": "#5B9BD5",
    "people": "#E8C44C",
    "toys": "#B07CC8",
    "vehicles": "#6BA3D5",
    "other": "#8B9A9E",
}

corr_df = corr_df.merge(sem_df, on="category", how="left")
corr_df["cdi_semantic"] = corr_df["cdi_semantic"].fillna("other")
corr_df["dot_color"] = corr_df["cdi_semantic"].map(lambda s: CDI_SEMANTIC_COLORS.get(s, CDI_SEMANTIC_COLORS["other"]))

# Marker size scales with per-file precision-valid exemplar count.
nmin, nmax = corr_df["n_valid_exemplars"].min(), corr_df["n_valid_exemplars"].max()
if np.isclose(nmin, nmax):
    corr_df["dot_size"] = 70.0
else:
    corr_df["dot_size"] = 40.0 + (corr_df["n_valid_exemplars"] - nmin) / (nmax - nmin) * 220.0

def add_corr_panel(ax, df, xcol, ycol, title, xlabel, ylabel):
    ax.scatter(
        df[xcol],
        df[ycol],
        c=df["dot_color"],
        s=df["dot_size"],
        alpha=0.82,
        edgecolor="white",
        linewidth=0.5,
    )

    rho = df[[xcol, ycol]].corr(method="spearman").iloc[0, 1]
    ax.text(
        0.02,
        0.98,
        f"Spearman rho = {rho:.3f}",
        transform=ax.transAxes,
        va="top",
        ha="left",
        fontsize=11,
        fontweight="bold",
        bbox=dict(boxstyle="round,pad=0.25", facecolor="white", edgecolor="#BBBBBB", alpha=0.92),
    )

    ax.set_title(title, fontsize=16, fontweight="bold", pad=10)
    ax.set_xlabel(xlabel, fontsize=14, fontweight="bold")
    ax.set_ylabel(ylabel, fontsize=14, fontweight="bold")
    ax.tick_params(axis="both", labelsize=14)
    for lbl in ax.get_xticklabels() + ax.get_yticklabels():
        lbl.set_fontweight("bold")
    ax.grid(alpha=0.18, linewidth=0.6)
    # Room for point labels so annotations stay inside the axes.
    ax.margins(x=0.18, y=0.18)


def annotate_local_global_panel(ax, df, xcol, ycol, plot_a_categories, n_outliers=3):
    """Label Plot A montage categories and largest OLS residual outliers (all bold).

    Works for any scatter of two numeric columns in ``corr_df`` (cross-model or within-model).
    """
    cat_lower = df["category"].astype(str).str.strip().str.lower()
    plot_a_set = {str(c).strip().lower() for c in plot_a_categories}

    sub_a = df.loc[cat_lower.isin(plot_a_set)]
    offsets_a = [(10, 8), (10, -12), (-8, 10), (-10, -10), (14, 2), (-12, -14)]
    for j, row in enumerate(sub_a.itertuples(index=False)):
        ox, oy = offsets_a[j % len(offsets_a)]
        ax.annotate(
            row.category,
            (getattr(row, xcol), getattr(row, ycol)),
            xytext=(ox, oy),
            textcoords="offset points",
            fontsize=10,
            fontweight="bold",
            color="#141414",
            bbox=dict(boxstyle="round,pad=0.25", facecolor="white", edgecolor="#555555", alpha=0.93),
            arrowprops=dict(arrowstyle="-", color="#666666", lw=0.8, alpha=0.9),
            clip_on=True,
            zorder=12,
        )

    work = df.loc[~cat_lower.isin(plot_a_set)].copy()
    if len(work) <= n_outliers:
        return
    xv = work[xcol].to_numpy(dtype=float)
    yv = work[ycol].to_numpy(dtype=float)
    coef = np.polyfit(xv, yv, 1)
    pred = coef[0] * xv + coef[1]
    work["_abs_resid"] = np.abs(yv - pred)
    outliers = work.nlargest(n_outliers, "_abs_resid")
    offsets_o = [(12, -14), (-14, 12), (10, 14)]
    for j, row in enumerate(outliers.itertuples(index=False)):
        ox, oy = offsets_o[j % len(offsets_o)]
        ax.annotate(
            row.category,
            (getattr(row, xcol), getattr(row, ycol)),
            xytext=(ox, oy),
            textcoords="offset points",
            fontsize=9,
            fontweight="bold",
            color="#222222",
            bbox=dict(boxstyle="round,pad=0.2", facecolor="white", edgecolor="#999999", alpha=0.9),
            arrowprops=dict(arrowstyle="-", color="#888888", lw=0.75, alpha=0.9),
            clip_on=True,
            zorder=11,
        )


# plt.rcParams["font.family"] = "Lato"
plt.rcParams["font.sans-serif"] = ["DejaVu Sans", "Arial", "Liberation Sans"]
fig, axes = plt.subplots(2, 2, figsize=(18, 13), dpi=180)
fig.subplots_adjust(left=0.08, right=0.78, top=0.88, bottom=0.16, wspace=0.24, hspace=0.24)

add_corr_panel(
    axes[0, 0], corr_df,
    "global_clip", "global_dino",
    "(1) Cross-model global dispersion",
    "Global dispersion (CLIP)", "Global dispersion (DINOv3)",
)
add_corr_panel(
    axes[0, 1], corr_df,
    "knn_clip", "knn_dino",
    "(2) Cross-model local coherence",
    "Local coherence (CLIP)", "Local coherence (DINOv3)",
)
add_corr_panel(
    axes[1, 0], corr_df,
    "global_clip", "knn_clip",
    "(3) CLIP: global dispersion vs local coherence",
    "Global dispersion (CLIP)", "Local coherence (CLIP)",
)
add_corr_panel(
    axes[1, 1], corr_df,
    "global_dino", "knn_dino",
    "(4) DINOv3: global dispersion vs local coherence",
    "Global dispersion (DINOv3)", "Local coherence (DINOv3)",
)

# Highlight Plot A montage categories + OLS residual outliers (top: fewer extras to reduce clutter).
plot_a_csv = base_dir / "plotA_category_montages_low_to_high" / "plotA_selected_categories_low_to_high_variability.csv"
if plot_a_csv.exists():
    plot_a_categories = pd.read_csv(plot_a_csv)["category"].astype(str).str.strip().tolist()
else:
    plot_a_categories = ["watch", "shower", "bird", "cup", "book"]
# Label pony as well (joint exemplars in notebook 02); dedupe if ever in CSV.
plot_a_categories = list(dict.fromkeys(plot_a_categories + ["pony"]))

annotate_local_global_panel(axes[0, 0], corr_df, "global_clip", "global_dino", plot_a_categories, n_outliers=2)
annotate_local_global_panel(axes[0, 1], corr_df, "knn_clip", "knn_dino", plot_a_categories, n_outliers=2)
annotate_local_global_panel(axes[1, 0], corr_df, "global_clip", "knn_clip", plot_a_categories, n_outliers=3)
annotate_local_global_panel(axes[1, 1], corr_df, "global_dino", "knn_dino", plot_a_categories, n_outliers=3)

# Bottom legend: CDI semantic (Spearman rho is drawn inside each subplot by add_corr_panel).
semantic_present = [s for s in CDI_SEMANTIC_ORDER if s in set(corr_df["cdi_semantic"])]
semantic_handles = [
    plt.Line2D([0], [0], marker="o", color="none", markerfacecolor=CDI_SEMANTIC_COLORS[s], markeredgecolor="none", markersize=8, label=s)
    for s in semantic_present
]
semantic_legend = fig.legend(
    handles=semantic_handles,
    title="CDI semantic",
    loc="lower center",
    bbox_to_anchor=(0.5, 0.02),
    ncol=max(1, len(semantic_handles)),
    frameon=True,
    fontsize=11,
    title_fontsize=12,
)

# Exemplar-count size legend.
qvals = np.unique(np.round(np.quantile(corr_df["n_valid_exemplars"], [0.2, 0.5, 0.8])).astype(int))
size_handles = []
for q in qvals:
    if np.isclose(nmin, nmax):
        ms = (70.0 ** 0.5)
    else:
        s = 30.0 + (q - nmin) / (nmax - nmin) * 170.0
        ms = float(max(3.5, s ** 0.5))
    size_handles.append(plt.Line2D([0], [0], marker="o", color="none", markerfacecolor="#777777", markeredgecolor="none", markersize=ms, label=f"n={q}"))

size_legend = fig.legend(
    handles=size_handles,
    title="Valid Exemplars",
    loc="upper left",
    bbox_to_anchor=(0.80, 0.38),
    frameon=True,
    fontsize=11,
    title_fontsize=12,
)

fig.suptitle(
    "CCN 2026: global dispersion and local coherence (CLIP and DINOv3)",
    fontsize=20,
    fontweight="bold",
    y=0.97,
)

panel_png = out_dir / "ccn2026_variability_correlation_2x2_semantic_size.png"
panel_pdf = out_dir / "ccn2026_variability_correlation_2x2_semantic_size.pdf"
fig.savefig(panel_png, bbox_inches="tight")
fig.savefig(panel_pdf, bbox_inches="tight", format="pdf")
plt.show()

print(f"Saved panel PNG: {panel_png}")
print(f"Saved panel PDF: {panel_pdf}")
print(f"Saved combined table: {combined_path}")
print(f"Categories retained after per-file precision filter: {len(corr_df)}")

## THINGS vs BabyView (original per-exemplar embeddings)

Load the rerun CSVs from `plotC_knn_diversity_outputs/original_embeddings_rerun/` and visualize cross-dataset agreement using the same (non-inverted) direction as the previous panel:
- `mean_knn_dist` (higher = more local variability)
- `global_dispersion` (higher = more spread around centroid)
- `local_over_global_noninverted` (`mean_knn_dist / global_dispersion`)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rerun_dir = Path(out_dir) / "original_embeddings_rerun"
category_set = "valid129"
k_used = 5


def _load_compare_df(embedding: str) -> pd.DataFrame:
    path = rerun_dir / f"things_vs_bv_{embedding}_local_global_k{k_used}_{category_set}.csv"
    if not path.exists():
        raise FileNotFoundError(f"Missing comparison CSV: {path}")
    df = pd.read_csv(path)
    # Use non-inverted measures to stay direction-consistent with the previous section.
    df["things_local_over_global_noninverted"] = (
        df["things_mean_knn_dist"] / df["things_global_dispersion"]
    )
    df["bv_local_over_global_noninverted"] = (
        df["bv_mean_knn_dist"] / df["bv_global_dispersion"]
    )
    df = df.dropna(
        subset=[
            "things_mean_knn_dist",
            "bv_mean_knn_dist",
            "things_global_dispersion",
            "bv_global_dispersion",
            "things_local_over_global_noninverted",
            "bv_local_over_global_noninverted",
        ]
    ).copy()
    return df


def _scatter_compare(df: pd.DataFrame, embedding: str, x_col: str, y_col: str, x_label: str, y_label: str):
    corr = df[[x_col, y_col]].corr().iloc[0, 1]
    fig, ax = plt.subplots(figsize=(6.2, 5.2), dpi=160)
    ax.scatter(df[x_col], df[y_col], s=26, alpha=0.78, edgecolor="none")

    # Identity line gives a quick sense of THINGS vs BV magnitude difference.
    lo = min(df[x_col].min(), df[y_col].min())
    hi = max(df[x_col].max(), df[y_col].max())
    ax.plot([lo, hi], [lo, hi], "--", linewidth=1.1, color="gray", alpha=0.8)

    ax.set_xlabel(x_label)
    ax.set_ylabel(y_label)
    ax.set_title(f"{embedding.upper()}  |  n={len(df)}  |  r={corr:.3f}")
    ax.grid(alpha=0.2)
    plt.show()


for emb in ["clip", "dinov3"]:
    cmp_df = _load_compare_df(emb)
    display(cmp_df[[
        "category",
        "things_mean_knn_dist", "bv_mean_knn_dist",
        "things_global_dispersion", "bv_global_dispersion",
        "things_local_over_global_noninverted", "bv_local_over_global_noninverted",
    ]].head(10))

    _scatter_compare(
        cmp_df,
        emb,
        x_col="things_mean_knn_dist",
        y_col="bv_mean_knn_dist",
        x_label="THINGS local variability (mean kNN dist)",
        y_label="BabyView local variability (mean kNN dist)",
    )

    _scatter_compare(
        cmp_df,
        emb,
        x_col="things_global_dispersion",
        y_col="bv_global_dispersion",
        x_label="THINGS global dispersion",
        y_label="BabyView global dispersion",
    )

    _scatter_compare(
        cmp_df,
        emb,
        x_col="things_local_over_global_noninverted",
        y_col="bv_local_over_global_noninverted",
        x_label="THINGS local/global ratio (non-inverted)",
        y_label="BabyView local/global ratio (non-inverted)",
    )

In [ ]:
# THINGS-only 2x2 panel (same style as CCN 2026 semantic-size panel)
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

base_dir = Path('.').resolve()
out_dir = Path(out_dir)
out_dir.mkdir(parents=True, exist_ok=True)

rerun_dir = out_dir / "original_embeddings_rerun"
category_set = "valid129"
k_used = 5

clip_things_path = rerun_dir / f"things_clip_local_global_k{k_used}_{category_set}.csv"
dino_things_path = rerun_dir / f"things_dinov3_local_global_k{k_used}_{category_set}.csv"

for p in [clip_things_path, dino_things_path]:
    if not p.exists():
        raise FileNotFoundError(f"Required THINGS rerun file not found: {p}")

clip_things = pd.read_csv(clip_things_path)
dino_things = pd.read_csv(dino_things_path)

things_corr_df = clip_things[["category", "mean_knn_dist", "global_dispersion", "n_exemplars"]].rename(
    columns={
        "mean_knn_dist": "knn_clip",
        "global_dispersion": "global_clip",
        "n_exemplars": "n_exemplars_clip",
    }
).merge(
    dino_things[["category", "mean_knn_dist", "global_dispersion", "n_exemplars"]].rename(
        columns={
            "mean_knn_dist": "knn_dino",
            "global_dispersion": "global_dino",
            "n_exemplars": "n_exemplars_dino",
        }
    ),
    on="category",
    how="inner",
)

things_corr_df["n_exemplars"] = things_corr_df[["n_exemplars_clip", "n_exemplars_dino"]].mean(axis=1)
things_corr_df["n_valid_exemplars"] = (
    things_corr_df[["n_exemplars_clip", "n_exemplars_dino"]].min(axis=1).round().astype(int)
)
things_corr_df = things_corr_df.dropna(subset=["knn_clip", "knn_dino", "global_clip", "global_dino"]).copy()
if things_corr_df.empty:
    raise ValueError("No THINGS categories available to build the THINGS 2x2 panel.")

# CDI semantic mapping + colors (reuse if already defined above).
repo_root = base_dir.parent.parent
semantic_csv = repo_root / "data" / "long_tailed_dist_prop_included_categories.csv"
if not semantic_csv.exists():
    raise FileNotFoundError(f"CDI semantic mapping not found: {semantic_csv}")

sem_df = pd.read_csv(semantic_csv, usecols=["category", "cdi_semantic"]).dropna(subset=["category", "cdi_semantic"])
sem_df["category"] = sem_df["category"].astype(str).str.strip()
sem_df["cdi_semantic"] = sem_df["cdi_semantic"].astype(str).str.strip().str.lower()

if "CDI_SEMANTIC_ORDER" not in globals():
    CDI_SEMANTIC_ORDER = [
        "animals", "body_parts", "clothing", "food_drink", "furniture_rooms",
        "household", "outside", "people", "toys", "vehicles", "other",
    ]
if "CDI_SEMANTIC_COLORS" not in globals():
    CDI_SEMANTIC_COLORS = {
        "animals": "#4DB8A8",
        "body_parts": "#E87A5F",
        "clothing": "#9B7EC8",
        "food_drink": "#E8A54C",
        "furniture_rooms": "#6BAB7A",
        "household": "#D97B9E",
        "outside": "#5B9BD5",
        "people": "#E8C44C",
        "toys": "#B07CC8",
        "vehicles": "#6BA3D5",
        "other": "#8B9A9E",
    }

things_corr_df = things_corr_df.merge(sem_df, on="category", how="left")
things_corr_df["cdi_semantic"] = things_corr_df["cdi_semantic"].fillna("other")
things_corr_df["dot_color"] = things_corr_df["cdi_semantic"].map(
    lambda s: CDI_SEMANTIC_COLORS.get(s, CDI_SEMANTIC_COLORS["other"])
)

nmin, nmax = things_corr_df["n_valid_exemplars"].min(), things_corr_df["n_valid_exemplars"].max()
if np.isclose(nmin, nmax):
    things_corr_df["dot_size"] = 70.0
else:
    things_corr_df["dot_size"] = 40.0 + (things_corr_df["n_valid_exemplars"] - nmin) / (nmax - nmin) * 220.0


def _add_corr_panel(ax, df, xcol, ycol, title, xlabel, ylabel):
    ax.scatter(
        df[xcol],
        df[ycol],
        c=df["dot_color"],
        s=df["dot_size"],
        alpha=0.82,
        edgecolor="white",
        linewidth=0.5,
    )

    rho = df[[xcol, ycol]].corr(method="spearman").iloc[0, 1]
    ax.text(
        0.02,
        0.98,
        f"Spearman rho = {rho:.3f}",
        transform=ax.transAxes,
        va="top",
        ha="left",
        fontsize=11,
        fontweight="bold",
        bbox=dict(boxstyle="round,pad=0.25", facecolor="white", edgecolor="#BBBBBB", alpha=0.92),
    )

    ax.set_title(title, fontsize=16, fontweight="bold", pad=10)
    ax.set_xlabel(xlabel, fontsize=14, fontweight="bold")
    ax.set_ylabel(ylabel, fontsize=14, fontweight="bold")
    ax.tick_params(axis="both", labelsize=14)
    for lbl in ax.get_xticklabels() + ax.get_yticklabels():
        lbl.set_fontweight("bold")
    ax.grid(alpha=0.18, linewidth=0.6)
    ax.margins(x=0.18, y=0.18)


plt.rcParams["font.sans-serif"] = ["DejaVu Sans", "Arial", "Liberation Sans"]
fig, axes = plt.subplots(2, 2, figsize=(18, 13), dpi=180)
fig.subplots_adjust(left=0.08, right=0.78, top=0.88, bottom=0.16, wspace=0.24, hspace=0.24)

_add_corr_panel(
    axes[0, 0], things_corr_df,
    "global_clip", "global_dino",
    "(1) THINGS cross-model global dispersion",
    "Global dispersion (THINGS CLIP)", "Global dispersion (THINGS DINOv3)",
)
_add_corr_panel(
    axes[0, 1], things_corr_df,
    "knn_clip", "knn_dino",
    "(2) THINGS cross-model local variability",
    "Local variability (THINGS CLIP; mean kNN dist)", "Local variability (THINGS DINOv3; mean kNN dist)",
)
_add_corr_panel(
    axes[1, 0], things_corr_df,
    "global_clip", "knn_clip",
    "(3) THINGS CLIP: global vs local variability",
    "Global dispersion (THINGS CLIP)", "Local variability (THINGS CLIP; mean kNN dist)",
)
_add_corr_panel(
    axes[1, 1], things_corr_df,
    "global_dino", "knn_dino",
    "(4) THINGS DINOv3: global vs local variability",
    "Global dispersion (THINGS DINOv3)", "Local variability (THINGS DINOv3; mean kNN dist)",
)

semantic_present = [s for s in CDI_SEMANTIC_ORDER if s in set(things_corr_df["cdi_semantic"])]
semantic_handles = [
    plt.Line2D([0], [0], marker="o", color="none", markerfacecolor=CDI_SEMANTIC_COLORS[s], markeredgecolor="none", markersize=8, label=s)
    for s in semantic_present
]
fig.legend(
    handles=semantic_handles,
    title="CDI semantic",
    loc="lower center",
    bbox_to_anchor=(0.5, 0.02),
    ncol=max(1, len(semantic_handles)),
    frameon=True,
    fontsize=11,
    title_fontsize=12,
)

qvals = np.unique(np.round(np.quantile(things_corr_df["n_valid_exemplars"], [0.2, 0.5, 0.8])).astype(int))
size_handles = []
for q in qvals:
    if np.isclose(nmin, nmax):
        ms = (70.0 ** 0.5)
    else:
        s = 30.0 + (q - nmin) / (nmax - nmin) * 170.0
        ms = float(max(3.5, s ** 0.5))
    size_handles.append(
        plt.Line2D([0], [0], marker="o", color="none", markerfacecolor="#777777", markeredgecolor="none", markersize=ms, label=f"n={q}")
    )

fig.legend(
    handles=size_handles,
    title="Exemplars per category",
    loc="upper left",
    bbox_to_anchor=(0.80, 0.38),
    frameon=True,
    fontsize=11,
    title_fontsize=12,
)

fig.suptitle(
    "CCN 2026: THINGS global dispersion and local variability (CLIP and DINOv3)",
    fontsize=20,
    fontweight="bold",
    y=0.97,
)

things_panel_png = out_dir / "ccn2026_variability_correlation_2x2_semantic_size_things.png"
things_panel_pdf = out_dir / "ccn2026_variability_correlation_2x2_semantic_size_things.pdf"
things_table_csv = out_dir / "ccn2026_local_global_extreme_categories_things_clip_dino.csv"

fig.savefig(things_panel_png, bbox_inches="tight")
fig.savefig(things_panel_pdf, bbox_inches="tight", format="pdf")
things_corr_df.to_csv(things_table_csv, index=False)
plt.show()

print(f"Saved THINGS panel PNG: {things_panel_png}")
print(f"Saved THINGS panel PDF: {things_panel_pdf}")
print(f"Saved THINGS combined table: {things_table_csv}")
print(f"THINGS categories retained: {len(things_corr_df)}")